In [ ]:
%pip install koreanize-matplotlib

In [ ]:
# 1. 수집
# 2. 탐색
# 2-1) 데이터를 읽고, 구조(스키마)확인, 자료형

In [ ]:
import koreanize_matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# pandas, seaborn 라이브러리 사용 옵션 설정
pd.set_option("display.max_columns", 100) #  최대 100개 열까지 표시한다.
pd.set_option("display.max_rows", 100) # 최대 100개 행까지 표시한다.
pd.set_option("display.width", 140) # 표의 출력 너비를 140자로 설정해 줄바꿈을 줄인다.
# seaborn 그래프의 기본 모양을 흰 배경과 격자선으로 설정하고, 노트북에 맞는 글자·요소 크기를 사용한다.
sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
df = pd.read_csv('train.csv')
# df.head(5)
df.head(5).T

In [ ]:
df.shape

In [ ]:
# duplicated() 행->이전에 동일한 행이 존재하면 True 아니면 Fasle -> sum()집계하면 True 개수
df.duplicated().sum()

In [ ]:
# 컬럼의 이름만 중복확인
df.columns.duplicated().sum()

In [ ]:
# 자료형
df.info()

In [ ]:
# df[조건][열이름]
# df.loc[조건, [열이름]]
df[['PoolQC', 'PoolArea']]

display(df.loc[df['PoolQC'].isna(), ['PoolQC', 'PoolArea']]) # df[df['PoolQC'].isna()]['PoolQC', 'PoolArea']
display(df.loc[df['PoolQC'].notna(), ['PoolQC', 'PoolArea']])

In [ ]:
df.dtypes.value_counts()

In [ ]:
# 변수 탐색 + 품질 검사(논리 검증)

In [ ]:
# 고유값
# df[컬럼].nunique() 값
df['PoolQC'].nunique(dropna=False)

# t1 = [df[c].nunique(dropna=False) for c in df.columns]

# 같은 표현
t1 = []
for c in df.columns: # for문을 많이 사용해야....
  t1.append(df[c].nunique(dropna=False))
t1

In [ ]:
df.dtypes.values

In [ ]:
# 고유값 품질 요약
unque_summary = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.values,
    'nunique(na)': [df[c].nunique(dropna=False) for c in df.columns], # t1
    'nunique(ratio)': [df[c].nunique(dropna=False)/len(df) for c in df.columns],
})
unque_summary

In [ ]:
# 개별 변수의 고유값 상세히 탐색 value_counts()

df['Id'].value_counts(dropna=False)

In [ ]:
for nc in df.select_dtypes('number').columns:
  display(df[nc].value_counts(dropna=False))


In [ ]:
for oc in df.select_dtypes('object').columns:
  display(df[oc].value_counts(dropna=False))

In [ ]:
for oc in df.select_dtypes('object').columns:
  print(len(df[oc].value_counts(dropna=False)))

In [ ]:
# 결측치 품질 요약
# 탐색, 확인이 목적 -> 대체/다른값으로 기록, ....

na_summary = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.values,
    'na(count)': df.isna().sum(),
}).sort_values('na(count)', ascending=False)

na_summary

In [ ]:
# 변수값 논리검정

In [ ]:
# 수치형 기술 통계량 점검
df.describe().T

In [ ]:
df.describe(include='object').T

In [ ]:
df.boxplot(['BsmtUnfSF','BsmtFinSF1','BsmtFinSF2', 'TotalBsmtSF'])

In [ ]:
df[df['BsmtFinSF1'] == 5644]

In [ ]:
# 예시1) 가격·면적의 음수 여부: SalePrice, LotArea, GrLivArea가 0보다 작은 행을 찾는다.
# 예시2) 연도 관계: YearBuilt가 YrSold보다 큰 집을 찾는다. 리모델링 연도(YearRemodAdd)가 판매 연도보다 늦은지도 확인한다.
# 예시3) 면적 합계: GrLivArea가 1stFlrSF + 2ndFlrSF + LowQualFinSF와 맞는지 확인한다.
# 예시4) 방·욕실 수: BedroomAbvGr, FullBath, GarageCars 등에 음수나 비현실적인 값이 있는지 확인한다.
# 예시5) 월 범위: MoSold가 1~12 범위인지 확인한다.
# 예시6) 식별자 중복: Id가 중복되거나 비어 있는지 확인한다.

In [ ]:
# 예시6) 식별자 중복: Id가 중복되거나 비어 있는지 확인한다.
# 전체1460개 고유값 1460 -> 식별자 검증

# RDB의 PK 검증
# 결측치 확인
print(df['Id'].isna().sum()) # 0 -> 식별자 결측치가 없다.
# 중복값 확인
print(df['Id'].duplicated().sum())

# 만약 위에서 하나라도 1이 있다면 조건식으로 행을 탐색
df[(df['Id'].isna() == True) | (df['Id'].duplicated() == True)]

In [ ]:
# 예시5) 월 범위: MoSold가 1~12 범위인지 확인한다.
df['MoSold'].unique() # 1 ~ 12 , 0,

In [ ]:
df[(df['MoSold']<1) | (df['MoSold']>12)]

In [ ]:
# 예시4) 방·욕실 수: BedroomAbvGr, FullBath, GarageCars 등에 음수나 비현실적인 값이 있는지 확인한다.

df[(df['BedroomAbvGr']<0) | (df['FullBath']<0) | (df['GarageCars']<0)]

In [ ]:
# 예시3) 면적 합계: GrLivArea가 1stFlrSF + 2ndFlrSF + LowQualFinSF와 맞는지 확인한다.
# df[['GrLivArea','1stFlrSF','2ndFlrSF','LowQualFinSF']].isna().sum()
# df[['GrLivArea','1stFlrSF','2ndFlrSF','LowQualFinSF']]

df[df['GrLivArea'] != (df['1stFlrSF']+df['2ndFlrSF']+df['LowQualFinSF'])]

# ~ 11/2

EDA 탐색 결과 보고서.md

1. 데이터셋 개요, 구조
2. 변수 - 변수 데이터 사전 참조
8. 데이터품질확인(논리검증)



---


3. 결측치
4. 이상치
5. 변수 분포(수치-히스토그램,박스플롯 / 범주-막대)
6. 타겟 분포, 분석
7. 상관관계

---

9. 전처리대한 의견
10. 아이디어 - 특성공학(요약변수, 파생변수)
11. 팀 결정사항

---

12. 탐색 문서를 바탕으로 1/n, 함께 전치리 ex) 결측치 처리

# 11/3 ~

In [ ]:
numeric_cols = df.select_dtypes('number').columns
category_cols = df.select_dtypes('object').columns

print(numeric_cols)
print(category_cols)

In [ ]:
# 히스토그램
# 분포, 값 집중,.... -> 탐색 내용정리 -> 조치 -> 로그변환 후 전/후 평가지표 확인, 제거, .....

df[numeric_cols].hist(figsize=(15, 12))

In [ ]:
# y 탐색
df['SalePrice'].hist()

In [ ]:
np.log1p(df['SalePrice']).hist()

In [ ]:
# 이상치 - 박스플롯
print(len(numeric_cols))
df[numeric_cols].plot(kind='box', subplots=True, layout=(8, 5), figsize=(20, 15))

In [ ]:
# 상관관계 x1값이 움지이면, x?값이 같이 움직이다. -> 변수간에 상관관계가 높으면 다중공선성 문제, X,y간 상관관계가 높다 중요

In [ ]:
cor = df.corr(numeric_only=True, method='pearson') # pearson기본값(연속형 수치) method='spearman'(순서가 있는 범주형 수치)

cor['SalePrice']

In [ ]:
plt.figure(figsize=(30, 25))
sns.heatmap(cor, annot=True, fmt='.2f')

In [ ]:
# 결측치
missing = df.isna().sum().sort_values(ascending=False)

missing[missing > 0]

None으로 처리하는 범주형 변수와 train 결측 수:

Alley: 1,369건. 골목 접근로가 없는 경우로 해석한다.

BsmtQual: 37건. 지하실이 없는 경우로 해석한다.TotalBsmtSF(지하실 면적) 0

BsmtCond: 37건. 지하실이 없는 경우로 해석한다.TotalBsmtSF가(지하실 면적) 0

BsmtExposure: 38건. 지하실이 없는 경우로 해석한다.TotalBsmtSF(지하실 면적) 0

BsmtFinType1: 37건. 지하실이 없는 경우로 해석한다.TotalBsmtSF(지하실 면적) 0

BsmtFinType2: 38건. 지하실이 없는 경우로 해석한다.TotalBsmtSF(지하실 면적) 0

FireplaceQu: 690건. 벽난로가 없는 경우로 해석한다.Fireplaces(벽난로 개수)0

GarageType: 81건. 차고가 없는 경우로 해석한다.GarageArea(차고 면적) 0

GarageFinish: 81건. 차고가 없는 경우로 해석한다.GarageArea(차고 면적) 0

GarageQual: 81건. 차고가 없는 경우로 해석한다.GarageArea(차고 면적) 0

GarageCond: 81건. 차고가 없는 경우로 해석한다.GarageArea(차고 면적) 0

PoolQC: 1,453건. 수영장이 없는 경우로 해석한다.PoolArea(수영장 면적) 0

Fence: 1,179건. 울타리가 없는 경우로 해석한다.

MiscFeature: 1,406건. 기타 시설이 없는 경우로 해석한다.

MasVnrType은 MasVnrArea와 같이 판단한다.

MasVnrType이 결측이고 MasVnrArea가 0이면 장식용 석재가 없는 것으로 보고
None으로 바꾼다.

MasVnrArea가 0이 아닌데 MasVnrType이 결측이면 임의로 종류를 추정하지 않고 Unknown으로 표시한다.

GarageYrBlt은 GarageType이 None인 행에서 결측이면 0으로 처리한다. 이는 차고가 없는 집에 차고 건축연도가 없다는 상태를 표현한다.

In [ ]:
# 전처리 의견에 대한 검정

print(len(df[df['BsmtQual'].isna() & (df['TotalBsmtSF'] == 0)]))
df[df['BsmtQual'].isna() & (df['TotalBsmtSF'] != 0)][['BsmtQual','TotalBsmtSF']]

In [ ]:
print(len(df[df['BsmtCond'].isna() & (df['TotalBsmtSF'] == 0)]))
df[df['BsmtCond'].isna() & (df['TotalBsmtSF'] != 0)][['BsmtCond','TotalBsmtSF']]

In [ ]:
print(len(df[df['BsmtFinType2'].isna() & (df['TotalBsmtSF'] == 0)]))
df[df['BsmtFinType2'].isna() & (df['TotalBsmtSF'] != 0)][['BsmtFinType2','TotalBsmtSF']]

In [ ]:
print(len(df[df['FireplaceQu'].isna() & (df['Fireplaces'] == 0)]))
df[df['FireplaceQu'].isna() & (df['Fireplaces'] != 0)][['FireplaceQu','TotalBsmtSF']]

In [ ]:
print(len(df[df['PoolQC'].isna() & (df['PoolArea'] == 0)]))
df[df['PoolQC'].isna() & (df['PoolArea'] != 0)][['PoolQC','PoolArea']]

In [ ]:
print(len(df[df['MasVnrType'].isna() & (df['MasVnrArea'] == 0)]))
df[df['MasVnrType'].isna() & (df['MasVnrArea'] != 0)][['MasVnrType','MasVnrArea']]

# 전처리 ~

In [ ]:
# 전치리전에 분할 먼저 진행
# 분할 -> 홀드아웃, 교차검증

# 변수 : Id(식별자), SalePrice(y)

X = df.drop(columns=['Id', 'SalePrice'])
y = df['SalePrice']

print(X.shape)
print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, root_mean_squared_error

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=1)

print(X_train.shape)
print(X_val.shape)
print(y_train.shape)
print(y_val.shape)

In [ ]:
# 결측치 처리 1) "None", "Unknown"
# 전체 결측치 수는
print('전:',X_train.isna().sum().sum())
print('전:',X_val.isna().sum().sum())

# 결측치를 "None"값으로 대처하는 컬럼들
na_cols =[
    'Alley','BsmtCond','BsmtCond','BsmtFinType1','FireplaceQu','GarageType','GarageFinish','GarageQual','GarageCond','PoolQC','Fence','MiscFeature'
]

for nc in na_cols:
  X_train[nc] = X_train[nc].fillna('None')
  X_val[nc] = X_val[nc].fillna('None')



In [ ]:
print('후:',X_train.isna().sum().sum())
print('후:',X_val.isna().sum().sum())

In [ ]:
t1 = X_train.isna().sum()
display(t1[t1>0])

t2 = X_val.isna().sum()
display(t2[t2>0])

In [ ]:
X_train.loc[X_train['BsmtExposure'].isna(), ['BsmtExposure', 'TotalBsmtSF']]

In [ ]:
# BsmtExposure, BsmtFinType2는 BsmtExposure결측치이고, TotalBsmtSF값이 0이면 "None", 아니면 "Unknown"
cond = X_train['BsmtExposure'].isna() & (X_train['TotalBsmtSF']==0)
X_train.loc[cond, 'BsmtExposure'] = "None"
X_train.loc[X_train['BsmtExposure'].isna(), 'BsmtExposure'] = "Unknown"


In [ ]:
display(X_train['BsmtExposure'].value_counts())

In [ ]:
cond = X_val['BsmtExposure'].isna() & (X_val['TotalBsmtSF']==0)
X_val.loc[cond, 'BsmtExposure'] = "None"

X_val.loc[X_val['BsmtExposure'].isna(), 'BsmtExposure'] = "Unknown"

In [ ]:

display(X_val['BsmtExposure'].value_counts())

In [ ]:
t1 = X_train.isna().sum()
display(t1[t1>0])

t2 = X_val.isna().sum()
display(t2[t2>0])

In [ ]:
cond = X_train['BsmtFinType2'].isna() & (X_train['TotalBsmtSF']==0)
X_train.loc[cond, 'BsmtFinType2'] = "None"
X_train.loc[X_train['BsmtFinType2'].isna(), 'BsmtFinType2'] = "Unknown"

cond = X_val['BsmtFinType2'].isna() & (X_val['TotalBsmtSF']==0)
X_val.loc[cond, 'BsmtFinType2'] = "None"

X_val.loc[X_val['BsmtFinType2'].isna(), 'BsmtFinType2'] = "Unknown"

In [ ]:
t1 = X_train.isna().sum()
display(t1[t1>0])

t2 = X_val.isna().sum()
display(t2[t2>0])

In [ ]:
X_train['BsmtFinType2'].value_counts()

In [ ]:
t1 = X_train.isna().sum()
display(t1[t1>0])

In [ ]:
cond = X_train['MasVnrType'].isna() & (X_train['MasVnrArea']==0)
X_train.loc[cond, 'MasVnrType'] = 'None'
X_train['MasVnrType'] = X_train['MasVnrType'].fillna('Unknown')

In [ ]:
cond = X_val['MasVnrType'].isna() & (X_val['MasVnrArea']==0)
X_val.loc[cond, 'MasVnrType'] = 'None'
X_val['MasVnrType'] = X_val['MasVnrType'].fillna('Unknown')

In [ ]:
t = X_train.isna().sum()
display(t[t>0])

In [ ]:
# GarageYrBlt은 GarageType이 None인 행에서 결측이면 0으로 처리한다. 이는 차고가 없는 집에 차고 건축연도가 없다는 상태를 표현한다.

# display(X_train.loc[X_train['GarageYrBlt'].isna(), ['GarageYrBlt','GarageType']])


cond = X_train['GarageYrBlt'].isna() & (X_train['GarageType'] == 'None')
X_train.loc[cond, 'GarageYrBlt'] = 0

In [ ]:
cond = X_val['GarageYrBlt'].isna() & (X_val['GarageType'] == 'None')
X_val.loc[cond, 'GarageYrBlt'] = 0

In [ ]:
t1 = X_train.isna().sum()
display(t1[t1>0])

t2 = X_val.isna().sum()
display(t2[t2>0])

In [ ]:
# 결측치 처리 2) 계산된 값으로 처리(평균/중앙값, 회귀결과값, 그룹의 평균/중앙값, .....)